In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [2]:
filtered_Cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Cohort_dfV1")
filtered_Control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Control_dfV1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Get the list of column names
column_names = filtered_Cohort.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns Cohort:", num_columns)
# Get the list of column names
column_names1 = filtered_Control.columns

# Count the number of columns
num_columns1 = len(column_names1)

# Display the number of columns
print("Number of columns Control:", num_columns1)

▸,:,


Number of columns Cohort: 2407
Number of columns Control: 2348


In [4]:
from pyspark.sql.functions import lit

# Find columns present in filtered_Control but not in filtered_Cohort
control_columns = filtered_Control.columns
cohort_columns = filtered_Cohort.columns
missing_columns = [col for col in control_columns if col not in cohort_columns]

# Add missing columns to filtered_Cohort with null values
for col in missing_columns:
    filtered_Cohort = filtered_Cohort.withColumn(col, lit(None))

▸,:,


In [5]:
from pyspark.sql.functions import lit

# Find columns present in filtered_Cohort but not in filtered_Control
cohort_columns = filtered_Cohort.columns
control_columns = filtered_Control.columns
missing_columns = [col for col in cohort_columns if col not in control_columns]

# Add missing columns to filtered_Cohort with null values
for col in missing_columns:
    filtered_Control = filtered_Control.withColumn(col, lit(None))

▸,:,


In [6]:
# Get the list of column names
column_names = filtered_Cohort.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns Cohort:", num_columns)
# Get the list of column names
column_names1 = filtered_Control.columns

# Count the number of columns
num_columns1 = len(column_names1)

# Display the number of columns
print("Number of columns Control:", num_columns1)

▸,:,


Number of columns Cohort: 2408
Number of columns Control: 2408


In [9]:
filtered_Cohort = filtered_Cohort.drop("stratification_key")

▸,:,


In [11]:
filtered_Control = filtered_Control.drop("stratification_key")

▸,:,


In [10]:
# Print column names and their corresponding indices
for index, col_name in enumerate(filtered_Cohort.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column MedicalHistory has index 8
Column Z21 has index 9
Column M19 has index 10
Column S68 has index 11
Column Y30 has index 12
Column B05 has index 13
Column A23 has index 14
Column H82 has index 15
Column V89 has index 16
Column I31 has index 17
Column V72 has index 18
Column R16 has index 19
Column Q61 has index 20
Column O12 has index 21
Column X76 has index 22
Column Z12 has index 23
Column S39 has index 24
Column L65 has index 25
Column F25 has index 26
Column G12 has index 27
Column E02 has index 28
Column X04 has index 29
Column B79 has index 30
Column F32 has index 31
Column M54 has index 32
Column B34 has index 33
Column S60 has index 34
Column Z19 has index 35
Column T36 has index 36
Column E44 has index 37
Column E83 has index 38
Colu

In [12]:
# Print column names and their corresponding indices
for index, col_name in enumerate(filtered_Control.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column TBI_date has index 2
Column latest_diagdate has index 3
Column age_of_TBI_diagnosis has index 4
Column race has index 5
Column gender has index 6
Column MedicalHistory has index 7
Column M19 has index 8
Column S68 has index 9
Column Z21 has index 10
Column Y30 has index 11
Column A23 has index 12
Column B05 has index 13
Column H82 has index 14
Column V89 has index 15
Column R16 has index 16
Column I31 has index 17
Column Q61 has index 18
Column V72 has index 19
Column O12 has index 20
Column X76 has index 21
Column S39 has index 22
Column Z12 has index 23
Column L65 has index 24
Column X04 has index 25
Column F25 has index 26
Column E02 has index 27
Column G12 has index 28
Column B79 has index 29
Column M54 has index 30
Column S60 has index 31
Column F32 has index 32
Column B34 has index 33
Column E44 has index 34
Column T36 has index 35
Column E83 has index 36
Column Q65 has index 37
Column R71 has index 38
Column Z19 has

In [13]:
import pandas as pd
import math
from pyspark.sql.functions import col
# Define the function to split the DataFrame
def split_df(df, num_split):
    total_columns = len(df.columns)
    first_column = df.columns[0]  # Get the name of the first column
    
    splitted = []
    
    num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
    for i in range(num_split):
        start = 1 + i * num_columns_per_split  # Start from the second column
        end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
        split_columns = [first_column] + list(df.columns[start:end])
        split_df = df[split_columns]
        splitted.append(split_df)
        
    return splitted

▸,:,


In [14]:
# Select columns from index 10 to 2340
columns_to_replace = filtered_Cohort.columns[9:2337]
# Replace null values with zeros in the selected columns using na.fill
filtered_Cohort = filtered_Cohort.fillna(0, subset=columns_to_replace)

# Split the DataFrame into 10 parts
filtered_Cohort_split = split_df(filtered_Cohort, 5)

▸,:,


In [15]:
# Join the split DataFrames into a single DataFrame
filtered_Cohort = filtered_Cohort_split[0]

for i in range(1, len(filtered_Cohort_split)):
    print(f"Joining DataFrame {i+1}...")
    filtered_Cohort = filtered_Cohort.join(filtered_Cohort_split[i], on='personid', how='inner')

▸,:,


Joining DataFrame 2...
Joining DataFrame 3...
Joining DataFrame 4...
Joining DataFrame 5...


In [17]:
filtered_Cohort.show(3, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+----------+-------------------------+-------------------------+--------------------+--------------------+-----+------+--------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

<IPython.core.display.Javascript object>

In [18]:
filtered_Cohort.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (n

In [19]:
filtered_Cohort = filtered_Cohort.drop("latest_diagdate")

▸,:,


In [20]:
filtered_Cohort.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Cohort_V1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from pyspark.sql.functions import col

# Get the column names from one of the DataFrames
column_names = filtered_Cohort.columns

# Reorder the columns in filtered_Control to match the order in filtered_Cohort
filtered_Control_reordered = filtered_Control.select(*column_names)

▸,:,


In [26]:
# Print column names and their corresponding indices
for index, col_name in enumerate(filtered_Control_reordered.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column MedicalHistory has index 8
Column Z21 has index 9
Column M19 has index 10
Column S68 has index 11
Column Y30 has index 12
Column B05 has index 13
Column A23 has index 14
Column H82 has index 15
Column V89 has index 16
Column I31 has index 17
Column V72 has index 18
Column R16 has index 19
Column Q61 has index 20
Column O12 has index 21
Column X76 has index 22
Column Z12 has index 23
Column S39 has index 24
Column L65 has index 25
Column F25 has index 26
Column G12 has index 27
Column E02 has index 28
Column X04 has index 29
Column B79 has index 30
Column F32 has index 31
Column M54 has index 32
Column B34 has index 33
Column S60 has index 34
Column Z19 has index 35
Column T36 has index 36
Column E44 has index 37
Column E83 has index 38
Colu

In [25]:
# Get the list of column names
column_names = filtered_Cohort.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns Cohort:", num_columns)
# Get the list of column names
column_names1 = filtered_Control_reordered.columns

# Count the number of columns
num_columns1 = len(column_names1)

# Display the number of columns
print("Number of columns Control:", num_columns1)

▸,:,


Number of columns Cohort: 2406
Number of columns Control: 2406


In [24]:
filtered_Control_reordered.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: null (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: null (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (nulla

In [27]:
import pandas as pd
import math
from pyspark.sql.functions import col
# Define the function to split the DataFrame
def split_df(df, num_split):
    total_columns = len(df.columns)
    first_column = df.columns[0]  # Get the name of the first column
    
    splitted = []
    
    num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
    for i in range(num_split):
        start = 1 + i * num_columns_per_split  # Start from the second column
        end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
        split_columns = [first_column] + list(df.columns[start:end])
        split_df = df[split_columns]
        splitted.append(split_df)
        
    return splitted

▸,:,


In [28]:
# Select columns from index 10 to 2340
columns_to_replace = filtered_Control_reordered.columns[9:2337]
# Replace null values with zeros in the selected columns using na.fill
filtered_Control_reordered = filtered_Control_reordered.fillna(0, subset=columns_to_replace)

# Split the DataFrame into 10 parts
filtered_Control_split = split_df(filtered_Control_reordered, 5)

▸,:,


In [29]:
# Join the split DataFrames into a single DataFrame
filtered_Control_reordered = filtered_Control_split[0]

for i in range(1, len(filtered_Control_split)):
    print(f"Joining DataFrame {i+1}...")
    filtered_Control_reordered = filtered_Control_reordered.join(filtered_Control_split[i], on='personid', how='inner')

▸,:,


Joining DataFrame 2...
Joining DataFrame 3...
Joining DataFrame 4...
Joining DataFrame 5...


In [31]:
from pyspark.sql.types import NullType, StringType
# Print the schema before conversion
print("Schema before conversion:")
filtered_Control_reordered.printSchema()

# Iterate over the schema and cast NullType columns to StringType
for field in filtered_Control_reordered.schema.fields:
    if isinstance(field.dataType, NullType):
        filtered_Control_reordered = filtered_Control_reordered.withColumn(field.name, filtered_Control_reordered[field.name].cast(StringType()))

# Print the schema after conversion
print("Schema after conversion:")
filtered_Control_reordered.printSchema()

# Show the DataFrame
filtered_Control_reordered.show(truncate=False)

▸,:,


Schema before conversion:
root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: null (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: null (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = t

Schema after conversion:
root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: string (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+-------------------------+--------------------+--------------------+----------+------+--------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

<IPython.core.display.Javascript object>

In [32]:
filtered_Control_reordered.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Control_V1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = filtered_Cohort.rdd.getNumPartitions()
filtered_Cohort_repart = filtered_Cohort.repartition(reparNum)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = filtered_Control_reordered.rdd.getNumPartitions()
filtered_Control_reordered_repart = filtered_Control_reordered.repartition(reparNum)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# Stack the two DataFrames vertically
stacked_df = filtered_Cohort_repart.union(filtered_Control_reordered_repart)

▸,:,


In [36]:
print(stacked_df.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

116072


<IPython.core.display.Javascript object>

In [37]:
# Get the list of column names
column_names = stacked_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 2406


In [38]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = stacked_df.rdd.getNumPartitions()
stacked_df_repart = stacked_df.repartition(reparNum)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
# stacked_df.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Stacked_CCSS')
# stacked_df_repart.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Stacked_CCSS_1')
stacked_df_repart.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Stacked_CCSS_D1.1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
filtered_Cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Cohort_I1")

In [ ]:
filtered_Cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Cohort_I1")